In [3]:
%matplotlib inline
import sys
import os
sys.path.append(os.path.abspath(".."))
import importlib

from functions import color_palettes
importlib.reload(color_palettes)
from functions.color_palettes import *

from functions import utils
importlib.reload(utils)
from functions.utils import *

from functions import config
importlib.reload(config)
from functions.config import *

In [5]:
df_analysis_ready = pd.read_excel(merged_cgm_meal_path, index_col=0)
metadata=pd.read_excel(metadata_path)
metadata=metadata.rename(columns={'ID':'Participant_id'})
metadata=clean_dataframe(metadata, keep_case_cols='Participant_id')
cgm_features=pd.read_excel(cgm_extracted_features_path,index_col=0)

In [6]:
ppgr_df=compute_ppgr_metrics_all_meals(df_analysis_ready)

In [7]:
meal_features_df = (
    df_analysis_ready
    .sort_values("time_since_meal_min")
    .groupby("meal_id")
    .first()
    .reset_index()
)
final_meal_df = (
    meal_features_df
    .merge(ppgr_df, on="meal_id", how="left")
    .merge(metadata.drop(columns=["completed_12days", "completed_10days", "complete_cgm_data"]), on="Participant_id", how="left")
    .merge(cgm_features.drop(columns=["age", "sex"]), on="Participant_id", how="left")
)

In [ ]:
final_meal_df.to_excel(r'C:\Users\aoezek\Desktop\master_thesis\master_thesis\finalised_clean_notebooks\excels\models_data_ppgr.xlsx')

---
#### PREPROCESSING : for iAUC PREDICTION
---

In [8]:
encoder = LabelEncoder()
final_meal_df['sex'] = encoder.fit_transform(final_meal_df['sex']) #female 0 male 1

In [ ]:
final_meal_df['iauc'].describe()

---
#### FEATURE ENGINEERING
---

##### thesis visualisations
---


In [ ]:
gi = final_meal_df["glycemic_index"]
sns.histplot(gi, kde=True, color=color)
plt.title("Distribution of Glycemic Index", fontsize=12, pad=8)
plt.xlabel('glycemix_index', fontsize=12, labelpad=8)
plt.ylabel('Count', fontsize=12, labelpad=8)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.savefig('skewness_gi.png', dpi=400, bbox_inches='tight')
plt.show()
print("Skewness:", gi.skew())

In [ ]:
fiber_skew = final_meal_df["fiber_g"]
sns.histplot(fiber_skew, kde=True, color=color)
plt.title("Distribution of Fiber Content (in grams)", fontsize=12, pad=8)
plt.xlabel('fiber_g', fontsize=12, labelpad=8)
plt.ylabel('Count', fontsize=12, labelpad=8)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.savefig('skewness_fiber.png', dpi=400, bbox_inches='tight')
plt.show()
print("Skewness:", fiber_skew.skew())

In [ ]:
iauc = final_meal_df["iauc"]
sns.histplot(iauc, kde=True, color=color)
plt.title("Distribution of iAUC", fontsize=12, pad=8)
plt.xlabel('iAUC', fontsize=12, labelpad=8)
plt.ylabel('Count', fontsize=12, labelpad=8)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.savefig('skewness_iauc.png', dpi=400, bbox_inches='tight')
plt.show()
print("Skewness:", iauc.skew())

In [10]:
drop_cols = [
    "meal_id", "Participant_id", "datetime","meal_time", "date",
    "Glucose_Value", "glucose_norm", "time_since_meal_min",
    "peak_increment", "time_to_peak", "time_hour", "estimated_total_weight_g",
    "self_reported_sleep_duration","10day_glucose_avg","10day_glucose_SD",
    "PSQI_global_score"
]
cols_to_check = [c for c in final_meal_df.columns if c not in drop_cols]

---
##### Feature Engineering
---

In [ ]:
skewness = final_meal_df[cols_to_check].skew()
skewed_features = skewness[abs(skewness) > 1].sort_values(ascending=False)
print(skewed_features)

In [12]:
tuned_df = final_meal_df.copy()
#eliminating negligible meals and extreme (outlier responses)
tuned_df = tuned_df[tuned_df['carbs_g'] > 5]
tuned_df = tuned_df[tuned_df['total_calories'] > 50]
tuned_df = tuned_df[tuned_df['glycemic_index'] > 0]
upper = tuned_df['iauc'].quantile(0.99)
tuned_df = tuned_df[tuned_df['iauc'] < upper]


In [13]:
skewed_features = skewness[abs(skewness) > 1].sort_values(ascending=False)

for col in skewed_features.index:
    if skewed_features[col] > 1 and tuned_df[col].min() >= 0:
        tuned_df[col] = np.log1p(tuned_df[col])
    elif skewed_features[col] < -1:
        pt = PowerTransformer(method="yeo-johnson")  #features that might be -minus
        tuned_df[[col]] = pt.fit_transform(tuned_df[[col]])

In [14]:
#interaction features
tuned_df["carbs_x_baseline"] = tuned_df["carbs_g"] * tuned_df["Glucose_Value"]
tuned_df["carbs_x_acrophase"] = tuned_df["carbs_g"] * tuned_df["acrophase"]
tuned_df["carbs_x_glucoseSD"] = tuned_df["carbs_g"] * tuned_df["10day_glucose_SD"]

In [15]:
#< > [] etc signs cleaning
tuned_df=clean_feature_names(tuned_df)  

In [16]:
 # participant IDs to skip bc of less than 10 meals (specially for personalized models)
exclude_ids = ["AL59", "AL54"] 
tuned_df_filtered = tuned_df[~tuned_df["Participant_id"].isin(exclude_ids)]

In [17]:
#add sin_time cos_time meal time to acrophase (circadian) features
tuned_df_filtered = engineer_circadian_feature(tuned_df_filtered)
tuned_df = engineer_circadian_feature(tuned_df)
tuned_df_filtered['Participant_id'] = tuned_df_filtered['Participant_id'].str.replace('AL', 'P', regex=False)
tuned_df['Participant_id'] = tuned_df['Participant_id'].str.replace('AL', 'P', regex=False)
tuned_df.rename(columns = {"Glucose_Value":"Baseline_Glucose"}, inplace=True)
tuned_df_filtered.rename(columns = {"Glucose_Value":"Baseline_Glucose"}, inplace=True)

In [18]:
tuned_df_filtered.to_excel(r'C:\Users\aoezek\Desktop\master_thesis\master_thesis\finalised_clean_notebooks\excels\tuned_engineered_task1_models_ready_df.xlsx', index=False)
tuned_df.to_excel(r'C:\Users\aoezek\Desktop\master_thesis\master_thesis\finalised_clean_notebooks\excels\tuned_engineered_task1_models_ready_df_NOT_FILTERED.xlsx', index=False)